# Skin Disease Prediction Using Deep Learning
### B.Tech Minor Project-I • CSE-AIML
**Complete Training Pipeline: MobileNetV2 Transfer Learning with Zero Data Leakage**

This Google Colab notebook provides the complete, production-grade deep learning training pipeline on the **HAM10000** benchmark dataset (10,015 dermatoscopic images across 7 diagnostic categories) using a free **NVIDIA T4 GPU**.

---
#### 🔬 Clinical Scope & Academic Integrity Note:
1. **Clinical Scope**: HAM10000 covers **7 specific categories of pigmented skin lesions**. It is **not** a general-purpose classifier for all dermatological conditions (e.g., eczema, acne, psoriasis, or fungal infections).
2. **Zero Data Leakage**: In dermoscopy, multiple photos are frequently taken of the same physical lesion at different angles or follow-ups. Splitting purely by image causes severe data leakage. This notebook strictly partitions data by patient lesion ID (`lesion_id`).
3. **Severe Class Imbalance**: In HAM10000, Melanocytic Nevi (`nv`) represents ~67% of all samples, while Melanoma (`mel`) represents ~11% and Dermatofibroma (`df`) is ~1%. Unweighted training causes models to collapse toward predicting `nv`. This pipeline computes balanced inverse-frequency class weights to heavily penalize misclassification of malignant lesions.
4. **Flask App Compatibility**: The input resolution is `(224, 224, 3)` with an embedded rescaling layer matching `predict.py` and `predictor.py` in your Flask web application. The output classes strictly follow alphabetical order (`akiec`, `bcc`, `bkl`, `df`, `mel`, `nv`, `vasc`).

In [ ]:
# ==============================================================================
# Step 1: Verify Hardware Accelerator (NVIDIA T4 GPU)
# ==============================================================================
# Ensure your Colab runtime is set to: Runtime -> Change runtime type -> T4 GPU
!nvidia-smi

In [ ]:
# ==============================================================================
# Step 2: Install and Import Required Dependencies (Including Kaggle CLI)
# ==============================================================================
!pip install -q kaggle tensorflow scikit-learn pillow matplotlib seaborn pandas numpy

import os
import sys
import json
import random
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    accuracy_score
)

# Set reproducible random seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# 7 Standard Benchmark Diagnostic Classes in Alphabetical Order
CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
NUM_CLASSES = len(CLASSES)
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASSES)}
IDX_TO_CLASS = {i: c for i, c in enumerate(CLASSES)}

print("TensorFlow Version:", tf.__version__)
print("Target Classes (Alphabetical):", CLASSES)
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

### Step 3: Authorized HAM10000 Dataset Acquisition

#### Terms Acceptance & Kaggle Authentication (Zero Hardcoded Credentials):
1. **Terms Acceptance**: The HAM10000 dataset is distributed under the **Creative Commons Attribution-NonCommercial 4.0 International (CC BY-NC 4.0)** license. Visit the [HAM10000 Dataset page](https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000) and click **I Understand and Agree** to accept the dataset rules.
2. **API Token**: Go to your Kaggle profile $\rightarrow$ **Settings** $\rightarrow$ **API** $\rightarrow$ Click **Create New Token**. This downloads `kaggle.json`.
3. **In-Memory Upload**: Run the cell below. It uses `files.upload()` to accept your `kaggle.json` interactively in memory without committing credentials to version control.
*(Alternatively, if your dataset is already mounted in Google Drive, set `USE_GOOGLE_DRIVE = True`)*.

In [ ]:
# ==============================================================================
# Step 3: Secure Dataset Download and Extraction (Robust & Verified)
# ==============================================================================
from google.colab import files

DATA_DIR = Path("./ham10000_data")
ARCHIVE_DIR = Path("./ham10000_archive")

# Pre-create directories before any download or extraction operations
DATA_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)

USE_GOOGLE_DRIVE = False  # Set to True if mounting from Google Drive

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # Example path: update to your Drive directory if using Drive
    # drive_zip = Path("/content/drive/MyDrive/HAM10000/skin-cancer-mnist-ham10000.zip")
    # if not drive_zip.exists():
    #     raise FileNotFoundError(f"Archive not found on Google Drive: {drive_zip}")
    # !unzip -q -o {drive_zip} -d {DATA_DIR}
else:
    # Check if dataset has already been unzipped in current Colab session
    found_metadata = list(DATA_DIR.glob("**/HAM10000_metadata.csv"))
    found_images = list(DATA_DIR.glob("**/*.jpg"))
    
    if found_metadata and len(found_images) >= 10000:
        print(f"HAM10000 dataset is already unzipped and verified ({len(found_images):,} images). Ready.")
    else:
        print("=== KAGGLE API SETUP ===")
        print("Please upload your 'kaggle.json' file using the button below.")
        uploaded = files.upload()
        
        if "kaggle.json" not in uploaded:
            raise FileNotFoundError(
                "kaggle.json was not uploaded! "
                "Download it from: Kaggle -> Settings -> API -> 'Create New Token', then rerun this cell."
            )
            
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        with open(kaggle_dir / "kaggle.json", "wb") as f:
            f.write(uploaded["kaggle.json"])
        os.chmod(kaggle_dir / "kaggle.json", 0o600)
        print("Kaggle credentials configured securely with chmod 600.")
        
        # Download archive to verified existing directory
        print("Downloading HAM10000 dataset (~2.8 GB)... This takes ~30 seconds on Colab.")
        !kaggle datasets download -d kmader/skin-cancer-mnist-ham10000 -p {ARCHIVE_DIR}
        
        # Verify that archive downloaded successfully and is non-empty (> 1 GB)
        archive_zip = ARCHIVE_DIR / "skin-cancer-mnist-ham10000.zip"
        if not archive_zip.exists() or archive_zip.stat().st_size < 1_000_000_000:
            raise RuntimeError(
                "Kaggle download failed or archive file is incomplete/missing. Possible causes:\n"
                "  1. You have not accepted the dataset terms at: "
                "https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000\n"
                "  2. Invalid or expired kaggle.json API token.\n"
                "  3. Kaggle API rate limit or network interruption."
            )
        print(f"Archive verified: {archive_zip.stat().st_size / (1024**3):.2f} GB on disk.")
        
        # Extract archive with error handling
        print(f"Extracting archive into {DATA_DIR}...")
        try:
            with zipfile.ZipFile(archive_zip, 'r') as zip_ref:
                zip_ref.extractall(DATA_DIR)
            print("Extraction complete.")
        except Exception as e:
            raise RuntimeError(f"Failed to extract {archive_zip}: {e}")
            
        # Immediate post-extraction validation
        extracted_meta = list(DATA_DIR.glob("**/HAM10000_metadata.csv"))
        extracted_imgs = list(DATA_DIR.glob("**/*.jpg"))
        if not extracted_meta or len(extracted_imgs) < 10000:
            raise RuntimeError(
                f"Extraction verification failed: Found {len(extracted_meta)} metadata files and "
                f"{len(extracted_imgs)} images. Expected HAM10000_metadata.csv and 10,015 images."
            )
        print(f"Extraction verified: {extracted_meta[0].name} and {len(extracted_imgs):,} images found.")

In [ ]:
# ==============================================================================
# Step 4: Metadata Inspection, Path Mapping & Full File Integrity Verification
# ==============================================================================
# 1. Find metadata CSV file
metadata_candidates = list(DATA_DIR.glob("**/HAM10000_metadata.csv"))
if not metadata_candidates:
    raise FileNotFoundError("HAM10000_metadata.csv not found. Ensure extraction succeeded.")

METADATA_PATH = metadata_candidates[0]
print(f"Found metadata file at: {METADATA_PATH}")

df_meta = pd.read_csv(METADATA_PATH)
print(f"Total metadata records loaded: {len(df_meta):,}")
print(f"Unique physical lesions identified: {df_meta['lesion_id'].nunique():,}")

if len(df_meta) != 10015:
    raise ValueError(f"Expected exactly 10,015 metadata records, but found {len(df_meta)}.")

# 2. Scan directories across both HAM10000_images_part_1 and part_2
print("Scanning directories for image files...")
all_image_paths = list(DATA_DIR.glob("**/*.jpg"))
image_path_dict = {p.stem: str(p) for p in all_image_paths}
print(f"Total JPG images found on disk: {len(image_path_dict):,}")

# 3. Attach verified file path to DataFrame and check for any missing images
df_meta['image_path'] = df_meta['image_id'].map(image_path_dict)
missing_images = df_meta['image_path'].isna().sum()

# Do NOT silently continue with a partial dataset
if missing_images > 0:
    missing_samples = df_meta[df_meta['image_path'].isna()]['image_id'].head(10).tolist()
    raise RuntimeError(
        f"Dataset integrity failure: {missing_images} metadata records could not be matched "
        f"to physical image files on disk! Sample missing IDs: {missing_samples}. "
        "Cannot proceed with a partial or incomplete dataset."
    )

print("All 10,015 images matched 100% successfully to verified files on disk.")

# 4. Filter strictly for the 7 benchmark classes
df_meta = df_meta[df_meta['dx'].isin(CLASSES)].copy().reset_index(drop=True)
df_meta['label_idx'] = df_meta['dx'].map(CLASS_TO_IDX)

print("\n--- Benchmark Class Distribution ---")
class_counts = df_meta['dx'].value_counts()[CLASSES]
for c in CLASSES:
    cnt = class_counts.get(c, 0)
    pct = (cnt / len(df_meta)) * 100
    print(f"  {c:5s} ({CLASS_TO_IDX[c]}): {cnt:5d} images ({pct:5.2f}%)")

In [ ]:
# ==============================================================================
# Step 5: Lesion-Level Zero Data Leakage Partitioning
# ==============================================================================
def partition_dataset_by_lesion(df, train_ratio=0.70, val_ratio=0.15, test_ratio=0.15, seed=SEED):
    """
    Partitions the dataset by lesion_id per class to guarantee zero cross-split leakage.
    """
    random.seed(seed)
    train_images, val_images, test_images = [], [], []
    
    train_lesions_total = set()
    val_lesions_total = set()
    test_lesions_total = set()
    
    for c in CLASSES:
        class_df = df[df['dx'] == c]
        unique_lesions = class_df['lesion_id'].unique().tolist()
        random.shuffle(unique_lesions)
        
        n = len(unique_lesions)
        n_test = max(1, round(n * test_ratio))
        n_val = max(1, round(n * val_ratio))
        n_train = n - n_val - n_test
        
        train_lesions = set(unique_lesions[:n_train])
        val_lesions = set(unique_lesions[n_train:n_train + n_val])
        test_lesions = set(unique_lesions[n_train + n_val:])
        
        # Strict lesion isolation assertion per class
        assert len(train_lesions & val_lesions) == 0, f"Leakage between Train & Val in class {c}!"
        assert len(train_lesions & test_lesions) == 0, f"Leakage between Train & Test in class {c}!"
        assert len(val_lesions & test_lesions) == 0, f"Leakage between Val & Test in class {c}!"
        
        train_lesions_total.update(train_lesions)
        val_lesions_total.update(val_lesions)
        test_lesions_total.update(test_lesions)
        
        train_images.extend(class_df[class_df['lesion_id'].isin(train_lesions)].index.tolist())
        val_images.extend(class_df[class_df['lesion_id'].isin(val_lesions)].index.tolist())
        test_images.extend(class_df[class_df['lesion_id'].isin(test_lesions)].index.tolist())
        
    # Global zero-leakage mathematical verification across entire benchmark
    assert len(train_lesions_total & val_lesions_total) == 0, "Global Train-Val lesion leakage!"
    assert len(train_lesions_total & test_lesions_total) == 0, "Global Train-Test lesion leakage!"
    assert len(val_lesions_total & test_lesions_total) == 0, "Global Val-Test lesion leakage!"
    
    train_df = df.loc[train_images].copy().sample(frac=1.0, random_state=seed).reset_index(drop=True)
    val_df = df.loc[val_images].copy().sample(frac=1.0, random_state=seed).reset_index(drop=True)
    test_df = df.loc[test_images].copy().reset_index(drop=True)  # Unshuffled for evaluation
    
    print("=" * 70)
    print("LESION-LEVEL ZERO DATA LEAKAGE CONFIRMED")
    print("=" * 70)
    print(f"Train Set:      {len(train_df):5d} images ({len(train_lesions_total):5d} distinct lesions)")
    print(f"Validation Set: {len(val_df):5d} images ({len(val_lesions_total):5d} distinct lesions)")
    print(f"Test Set:       {len(test_df):5d} images ({len(test_lesions_total):5d} distinct lesions)")
    print("Intersection (Train ∩ Val ∩ Test) = ∅ (0 overlap)")
    print("=" * 70)
    return train_df, val_df, test_df

train_df, val_df, test_df = partition_dataset_by_lesion(df_meta)

In [ ]:
# ==============================================================================
# Step 6: Balanced Class Weights & Optimized tf.data Pipelines
# ==============================================================================
# 1. Calculate balanced class weights
y_train_indices = train_df['label_idx'].values
raw_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=y_train_indices
)
class_weight_dict = {int(i): float(raw_weights[i]) for i in range(NUM_CLASSES)}

print("Balanced Class Weights (Penalizes majority 'nv' and prioritizes malignant 'mel'):")
for idx in sorted(class_weight_dict.keys()):
    print(f"  Class {idx} ({CLASSES[idx]:5s}): {class_weight_dict[idx]:.3f}")

# 2. Image loading and preprocessing matching predict.py / predictor.py in Flask app
IMG_HEIGHT, IMG_WIDTH = 224, 224
BATCH_SIZE = 32

def process_path_and_label(file_path, label):
    """
    Reads an image from disk, decodes JPEG, resizes to (224, 224),
    and scales pixels to [0.0, 1.0] matching predict.py in the Flask app.
    """
    img_bytes = tf.io.read_file(file_path)
    img = tf.io.decode_jpeg(img_bytes, channels=3)
    img = tf.image.resize(img, [IMG_HEIGHT, IMG_WIDTH], method='bilinear')
    img = tf.cast(img, tf.float32) / 255.0  # Normalized to [0.0, 1.0]
    one_hot_label = tf.one_hot(label, depth=NUM_CLASSES)
    return img, one_hot_label

def build_tf_dataset(df, is_training=False, batch_size=BATCH_SIZE):
    paths = df['image_path'].values
    labels = df['label_idx'].values
    
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    if is_training:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED)
        
    ds = ds.map(process_path_and_label, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size)
    ds = ds.prefetch(buffer_size=tf.data.AUTOTUNE)
    return ds

train_ds = build_tf_dataset(train_df, is_training=True)
val_ds = build_tf_dataset(val_df, is_training=False)
test_ds = build_tf_dataset(test_df, is_training=False)

print("\nOptimized tf.data pipelines assembled successfully.")

In [ ]:
# ==============================================================================
# Step 7: MobileNetV2 Transfer Learning Architecture Definition
# ==============================================================================
def build_mobilenet_v2_model(input_shape=(224, 224, 3), num_classes=NUM_CLASSES, dropout_rate=0.30):
    """
    Constructs Transfer Learning Model using MobileNetV2 with embedded rescaling and data augmentation.
    """
    inputs = tf.keras.Input(shape=input_shape, name="input_image")
    
    # Data Augmentation (only active during model training)
    augmented = layers.RandomFlip("horizontal")(inputs)
    augmented = layers.RandomRotation(0.1, fill_mode="reflect")(augmented)
    augmented = layers.RandomZoom((-0.1, 0.1), fill_mode="reflect")(augmented)
    augmented = layers.RandomContrast(0.1)(augmented)
    
    # Rescaling: maps [0.0, 1.0] from predict.py/predictor.py to [-1.0, 1.0] required by MobileNetV2
    rescaled = layers.Rescaling(scale=2.0, offset=-1.0, name="mobilenet_rescaling")(augmented)
    
    # Pre-trained MobileNetV2 backbone (initially frozen)
    base_model = MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = False
    
    features = base_model(rescaled, training=False)
    pooled = layers.GlobalAveragePooling2D(name="global_avg_pool")(features)
    normed = layers.BatchNormalization(name="batch_norm")(pooled)
    dense_1 = layers.Dense(256, activation="relu", name="dense_256")(normed)
    dropped = layers.Dropout(dropout_rate, name="dropout_head")(dense_1)
    outputs = layers.Dense(num_classes, activation="softmax", name="disease_prediction")(dropped)
    
    model = models.Model(inputs=inputs, outputs=outputs, name="SkinDisease_MobileNetV2")
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc", multi_label=True)
        ]
    )
    return model, base_model

model, base_model = build_mobilenet_v2_model()

trainable_count = sum(p.numpy().size for p in model.trainable_variables)
non_trainable_count = sum(p.numpy().size for p in model.non_trainable_variables)
print(f"Total Model Parameters:     {trainable_count + non_trainable_count:,}")
print(f"Initial Trainable Params:   {trainable_count:,}")
print(f"Initial Frozen Params:      {non_trainable_count:,}")

In [ ]:
# ==============================================================================
# Step 8: Two-Stage Model Training & Fine-Tuning
# ==============================================================================
MODEL_SAVE_PATH = "skin_disease_model.h5"

training_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

print("=" * 70)
print("STAGE 1: TRAINING TOP CLASSIFICATION HEAD (BACKBONE FROZEN)")
print("=" * 70)
history_stage1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    class_weight=class_weight_dict,
    callbacks=training_callbacks,
    verbose=1
)

print("\n" + "=" * 70)
print("STAGE 2: FINE-TUNING MOBILENETV2 UPPER LAYERS")
print("=" * 70)
# Unfreeze MobileNetV2 from layer 100 onwards for fine-grained dermatoscopic feature learning
base_model.trainable = True
for layer in base_model.layers[:100]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # Lower learning rate for fine-tuning
    loss="categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc", multi_label=True)]
)

history_stage2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weight_dict,
    callbacks=training_callbacks,
    verbose=1
)

print("\nTraining successfully concluded. Best model weights saved to:", MODEL_SAVE_PATH)

In [ ]:
# ==============================================================================
# Step 9: Reload Best Saved Model Checkpoint & Genuine Test-Set Evaluation
# ==============================================================================
print("=" * 70)
print("RELOADING BEST SAVED MODEL CHECKPOINT FOR TEST EVALUATION")
print("=" * 70)

# Ensure evaluation strictly evaluates the highest-performing validation-loss checkpoint
checkpoint_path = Path(MODEL_SAVE_PATH)
if not checkpoint_path.exists() or checkpoint_path.stat().st_size == 0:
    raise FileNotFoundError(f"Best model checkpoint not found or empty at: {MODEL_SAVE_PATH}")

best_model = tf.keras.models.load_model(str(checkpoint_path))
print(f"Successfully reloaded best saved checkpoint from: {checkpoint_path} ({checkpoint_path.stat().st_size / (1024**2):.2f} MB)")

# Run model prediction on holdout test set using the reloaded best model
y_true_indices = test_df['label_idx'].values
y_pred_probs = best_model.predict(test_ds, verbose=1)
y_pred_indices = np.argmax(y_pred_probs, axis=1)

# 1. Overall Test Metrics
test_acc = accuracy_score(y_true_indices, y_pred_indices)
try:
    test_auc = roc_auc_score(y_true_indices, y_pred_probs, multi_class='ovr')
except Exception:
    test_auc = None

print(f"\nTest Accuracy: {test_acc * 100:.2f}%")
if test_auc:
    print(f"Test ROC-AUC:  {test_auc:.4f}")

# 2. Per-Class Clinical Classification Report
print("\n--- Detailed Classification Report ---")
report_str = classification_report(
    y_true_indices,
    y_pred_indices,
    target_names=CLASSES,
    digits=4
)
print(report_str)

# 3. Confusion Matrix Plotting
cm = confusion_matrix(y_true_indices, y_pred_indices)

# 4. Per-Class Clinical Sensitivity and Specificity
print("\n--- Per-Class Sensitivity & Specificity ---")
for i, c in enumerate(CLASSES):
    tp = cm[i, i]
    fn = cm[i, :].sum() - tp
    fp = cm[:, i].sum() - tp
    tn = cm.sum() - (tp + fn + fp)
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    print(f"  {c:5s}: Sensitivity = {sens*100:5.2f}%, Specificity = {spec*100:5.2f}%")

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASSES,
    yticklabels=CLASSES
)
plt.title("HAM10000 Test Set Confusion Matrix (MobileNetV2)")
plt.xlabel("Predicted Diagnostic Label")
plt.ylabel("True Clinical Diagnosis")
plt.tight_layout()
plt.savefig("test_confusion_matrix.png", dpi=300)
plt.show()

In [ ]:
# ==============================================================================
# Step 10: Export Trained Model and model_info.json for Flask Web App
# ==============================================================================
import json
from google.colab import files

MODEL_FILE = Path(MODEL_SAVE_PATH)
INFO_FILE = Path("model_info.json")

# Generate metadata matching exact Flask requirements in model_loader.py / predict.py
model_info = {
    "project_title": "Skin Disease Prediction Using Deep Learning",
    "project_type": "B.Tech CSE-AIML Minor Project-I",
    "model_status": "trained",
    "model_architecture": "MobileNetV2 (Transfer Learning)",
    "target_input_shape": [224, 224, 3],
    "normalization": "[0.0, 1.0] with internal [-1.0, 1.0] MobileNetV2 scaling layer",
    "expected_weights_files": ["skin_disease_model.h5"],
    "benchmark_dataset": {
        "name": "HAM10000 (Human Against Machine with 10000 training images)",
        "source": "ISIC Archive / Harvard Dataverse",
        "total_images": 10015,
        "num_classes": 7
    },
    "test_metrics": {
        "accuracy": round(float(test_acc), 4),
        "roc_auc": round(float(test_auc), 4) if test_auc else None
    },
    "classes": [
        {
            "code": "akiec",
            "name": "Actinic Keratoses and Intraepithelial Carcinoma",
            "common_name": "Solar Keratosis / Pre-cancerous lesion",
            "type": "Pre-cancerous / Malignant potential"
        },
        {
            "code": "bcc",
            "name": "Basal Cell Carcinoma",
            "common_name": "Basal Cell Skin Cancer",
            "type": "Malignant"
        },
        {
            "code": "bkl",
            "name": "Benign Keratosis-like Lesions",
            "common_name": "Seborrheic Keratosis / Solar Lentigo",
            "type": "Benign"
        },
        {
            "code": "df",
            "name": "Dermatofibroma",
            "common_name": "Benign fibrous histiocytoma",
            "type": "Benign"
        },
        {
            "code": "mel",
            "name": "Melanoma",
            "common_name": "Malignant Melanoma",
            "type": "Malignant (High Risk)"
        },
        {
            "code": "nv",
            "name": "Melanocytic Nevi",
            "common_name": "Common Mole",
            "type": "Benign"
        },
        {
            "code": "vasc",
            "name": "Vascular Lesions",
            "common_name": "Angioma / Pyogenic Granuloma",
            "type": "Benign"
        }
    ]
}

with open(INFO_FILE, "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

# Explicit verification that both exported files exist and are non-empty
if not MODEL_FILE.exists() or MODEL_FILE.stat().st_size == 0:
    raise FileNotFoundError(f"Export verification failed: {MODEL_FILE} does not exist or is 0 bytes!")
if not INFO_FILE.exists() or INFO_FILE.stat().st_size == 0:
    raise FileNotFoundError(f"Export verification failed: {INFO_FILE} does not exist or is 0 bytes!")

print("=" * 70)
print("EXPORTED FILES VERIFIED SUCCESSFULLY")
print(f"  Model Weights:  {MODEL_FILE.resolve()} ({MODEL_FILE.stat().st_size / (1024**2):.2f} MB)")
print(f"  Model Metadata: {INFO_FILE.resolve()} ({INFO_FILE.stat().st_size} bytes)")
print("=" * 70)

print("\n--- Initiating Browser Downloads ---")
files.download(str(MODEL_FILE))
files.download(str(INFO_FILE))

print("\n" + "=" * 70)
print("NEXT STEPS FOR LOCAL FLASK WEB APPLICATION:")
print("=" * 70)
print("1. Move the downloaded 'skin_disease_model.h5' and 'model_info.json' into:")
print("   C:\\Users\\lavan\\.gemini\\antigravity\\scratch\\skin-disease-prediction\\model\\")
print("2. Start your local Flask app:")
print("   python app.py")
print("3. Navigate to http://127.0.0.1:5000/ to test live real-time clinical inference!")
print("=" * 70)